# 4. stdio与HTTP连接

第三节把加法和除法整理成 [计算服务脚本](servers/calculator_server.py)。当工具进入独立的服务进程，Client 怎样找到它？服务由谁启动，关闭 Client 后服务会不会一起结束？

本节复用同一份工具，分别使用 **stdio** 和 **Streamable HTTP**，用日志、进程编号（PID）与第二次连接观察完整生命周期。

学习目标：

1. 分清脚本连接与 URL 连接，以及服务进程由谁管理。
2. 解释两种 Transport、服务启动接口和 Client 调用的参数与返回值。
3. 实际启动、发现并调用工具，再关闭会话与服务。
4. 对比两种连接的工具 Schema、计算结果，以及适配后的 LangChain 工具。

全部自动实验使用当前电脑，不请求模型或外部服务，也不需要 API Key。HTTP 只监听 `127.0.0.1`；自动实验选择可用端口，结束时清理自己启动的进程和临时日志。选择项目 `.venv` 内核后，按顺序执行。

## 4.1 同一份工具，两种连接方式

![stdio 与 Streamable HTTP 的连接和进程](assets/13-stdio-and-http-topology.svg)

| 比较项 | stdio | Streamable HTTP |
|---|---|---|
| Client 的连接目标 | 服务脚本及其 Python 解释器 | 完整 MCP URL |
| 如何传递消息 | 服务端 stdin / stdout 与客户端相连的管道 | HTTP 请求和响应；需要时可使用流式响应 |
| 谁启动服务 | 通常由 Client 的 Transport 启动子进程 | 由服务运行者先启动，本节由 notebook 的启动器完成 |
| 是否需要 HTTP 端口 | 不需要 | 需要 host、port、path 对齐 |
| Client 会话结束 | 是否结束子进程由 keep_alive 决定 | 会话结束，服务进程继续运行 |
| 哪些内容可以相同 | 工具名称、输入 Schema、业务逻辑、计算结果 | 同左 |

Streamable HTTP 是 MCP 的一种标准传输方式，并不要求每个工具结果都逐字流式输出。客户端仍用 `list_tools()`、`call_tool()`；工具定义和业务参数不用改成另一套格式。

## 4.2 环境准备与进程内基准

除了已有的 FastMCP，本节用 Python 标准库启动和管理本地进程。`psutil` 已随当前 Jupyter 内核环境安装，仅用于查看进程是否存在，不属于 MCP 组件。

先记录脚本的进程内 Schema，之后两种连接都与它比较。`runpy` 的解释见 [第三节 3.7](03_用FastMCP创建自己的服务.ipynb)。

In [1]:
import asyncio
import os
import re
import runpy
import socket
import sys
import time
from contextlib import asynccontextmanager
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import psutil
from fastmcp import Client, FastMCP
from fastmcp.client.transports import PythonStdioTransport, StreamableHttpTransport
from langchain.mcp import MCPAdapter
from langchain_core.messages import ToolMessage

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 lang-chain-learn 项目或其子目录运行本笔记。")

SCRIPT_FILE = PROJECT_ROOT / "Chapter11_MCP" / "servers" / "calculator_server.py"
HTTP_HOST = "127.0.0.1"
MCP_PATH = "/mcp"

print("Python:", sys.executable)
for package in ["fastmcp", "mcp", "langchain", "psutil"]:
    print(f"{package}: {version(package)}")
print("计算服务脚本存在:", SCRIPT_FILE.is_file())

Python: /Users/skk/Developer/lang-chain-learn/.venv/bin/python
fastmcp: 4.1.0
mcp: 2.3.0
langchain: 1.4.3
psutil: 7.2.2
计算服务脚本存在: True


/var/folders/14/yh05p4nj2z70mjs568rzs9gh0000gn/T/ipykernel_72764/1860778055.py:16: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


In [2]:
script_objects = runpy.run_path(str(SCRIPT_FILE))
# 创建当前Server的客户端
async with Client(script_objects["mcp"]) as baseline_client:
    baseline_tools = await baseline_client.list_tools()

# 名字和input_schema的映射
baseline_schemas = {item.name: item.input_schema for item in baseline_tools}
# baseline_schemas = {item.name: item.input_schema for item in baseline_tools}
print("服务名称:", script_objects["mcp"].name)
print("基准工具:", sorted(baseline_schemas))
print("基准连接后 Client 已关闭:", not baseline_client.is_connected())

服务名称: Calculator
基准工具: ['add', 'divide']
基准连接后 Client 已关闭: True


## 4.3 服务脚本如何选择启动方式

本节在同一个脚本的启动入口中增加命令行参数，保留第三节的工具与默认 stdio 用法。`argparse` 是 Python 自带的命令行参数解析模块：解析 `--transport` 等参数后，再调用 `mcp.run(...)`。

在**项目根目录的终端**，两种启动命令分别是：

```bash
# stdio：等待协议客户端请求；手动观察后用 Ctrl+C 关闭
uv run python Chapter11_MCP/servers/calculator_server.py --transport stdio

# Streamable HTTP：监听本机 8000 端口，MCP 路径为 /mcp
uv run python Chapter11_MCP/servers/calculator_server.py --transport http --host 127.0.0.1 --port 8000 --path /mcp
```

自动实验会自行启动服务，不需要先执行这些命令。stdio 不会自动显示计算结果；HTTP 启动成功也只表示服务在监听，还需要客户端发现和调用工具。

### 4.3.1 FastMCP启动接口的参数与返回值

下面省略方法的 self。实际调用使用 mcp.run(...) 或 await mcp.run_async(...)：

```python
def run(transport=None, show_banner=None, **transport_kwargs) -> None: ...
async def run_async(transport=None, show_banner=None, **transport_kwargs) -> None: ...
```

| 参数 / 返回值 | 本节含义 |
|---|---|
| `transport` | `"stdio"` 使用管道；`"http"` 在 FastMCP 中表示 Streamable HTTP，也支持 `"streamable-http"` 别名。本节始终显式指定 |
| `show_banner` | 是否显示启动横幅；本节传 False，None 使用库配置 |
| `**transport_kwargs` | 转交给对应传输方式的启动参数；HTTP 使用下方的 host、port、path |
| `host` | 服务监听的地址；`127.0.0.1` 表示当前电脑 |
| `port` | 监听端口，整数；手动示例为 8000，自动实验使用可用端口 |
| `path` | MCP 的 HTTP 路径，例如 `"/mcp"`；它不是 Python 文件路径 |
| 返回值 | 服务正常结束后返回 None，不返回工具列表或计算结果；运行期间持续等待请求 |

脚本中的同步 `main()` 使用 `mcp.run(...)`。已有事件循环的异步代码可以使用 `await mcp.run_async(...)`，但它也会持续运行。本节在**服务子进程**里启动，所以 notebook 可以继续执行客户端代码；不要直接在 notebook 单元格中运行持续等待的服务入口。

### 4.3.2 为什么日志写到stderr

![stdio 协议流与日志流](assets/14-stdio-lifecycle-and-logs.svg)

服务脚本只在启动入口配置日志：

```python
logging.basicConfig(level=logging.INFO, stream=sys.stderr, ...)
```

`logger.info(...)` 因此写到 stderr；加法、除法只通过函数 `return` 返回计算值。stdio 的 stdout 是 MCP 协议通道，往里面 `print("正在计算")` 会混入普通文本，可能破坏消息解析。

HTTP 不靠 stdout 传递协议，但本脚本继续把日志写到 stderr。下面捕获实际日志，核对服务 PID 与工具执行。

## 4.4 stdio：由Client启动和关闭子进程

### 4.4.1 构造连接配置，再进入会话

`PythonStdioTransport` 的完整参数已在第三节 3.7 解释，这里结合实际启动再核对一次：

```python
PythonStdioTransport(
    script_path, args=None, env=None, cwd=None,
    python_cmd=sys.executable, keep_alive=None, log_file=None,
)
```

| 参数 | 本节传入内容与作用 |
|---|---|
| `script_path` | 计算脚本的 Path；构造时检查文件存在 |
| `args` | 脚本之后的命令行参数列表，这里是 `["--transport", "stdio"]` |
| `env` | 子进程额外环境变量，本计算服务不需要，使用 None |
| `cwd` | 子进程工作目录，传项目根目录字符串 |
| `python_cmd` | 启动脚本用的 Python，传 sys.executable |
| `keep_alive` | 当前 None 使用 True；先显式 False，退出连接时结束子进程 |
| `log_file` | 捕获服务端 stderr 的 Path 或文本文件对象，None 时使用当前 stderr |

构造表达式返回 Transport 对象，仅保存启动配置。`Client(transport, timeout=15, init_timeout=15)` 返回 Client 对象；进入 `async with` 才建立连接并完成协议初始化。

| Client接口 | 输入与返回值 |
|---|---|
| `Client(transport, timeout=None, init_timeout=None, ...)` | transport 为连接目标；数值超时以秒计，timeout 用于请求，init_timeout 用于初始化；返回 Client 对象 |
| `client.is_connected()` | 无业务参数，同步返回 bool；反映 Client 是否处于连接状态 |
| `await client.list_tools(max_pages=250, *, cache_mode="use")` | 分页上限和缓存策略可选；返回 MCP 工具描述列表 |
| `await client.call_tool(name, arguments=None, *, timeout=None, raise_on_error=True, ...)` | name 为工具名，arguments 为业务参数字典；等待后返回 CallToolResult。其余参数及结果字段见第三节 3.7.2 |
| `await client.close()` | 显式关闭 Client 与底层 Transport；等待后为 None，本节用于清理保留的 stdio 子进程 |

先定义两个日志观察函数，它们是本节的 Python 辅助函数，不是 MCP 接口。

In [3]:
def read_server_pid(log_path: Path) -> int:
    """从计算服务的启动日志取得 PID。"""
    log_text = log_path.read_text(encoding="utf-8")
    # 找到所有server_pid
    matches = re.findall(r"server_pid=(\d+)", log_text)
    if not matches:
        raise ValueError(f"日志中没有计算服务 PID：{log_text}")
    return int(matches[-1])


def calculator_log_lines(log_text: str) -> list[str]:
    """只展示计算脚本自己的日志，省略框架启动信息。"""
    return [line for line in log_text.splitlines() if line.startswith("calculator:")]

In [10]:
with TemporaryDirectory(prefix="mcp-stdio-") as log_directory:
    # 建立temp calculator-stderr.log
    stdio_log_path = Path(log_directory) / "calculator-stderr.log"
    # 创建子进程连接, 返回连接配置
    stdio_transport = PythonStdioTransport(
        script_path=SCRIPT_FILE,
        # 使用输入输出通信
        args=["--transport", "stdio"],
        python_cmd=sys.executable,
        cwd=str(PROJECT_ROOT),
        keep_alive=False,
        log_file=stdio_log_path,
    )
    stdio_client = Client(stdio_transport, timeout=15, init_timeout=15)
    print("进入前 Client 已连接:", stdio_client.is_connected())

    async with stdio_client:
        # 读取日志文件
        stdio_pid = read_server_pid(stdio_log_path)
        print("进入后 Client 已连接:", stdio_client.is_connected())
        print("服务 PID:",stdio_pid,"进程存在:",psutil.pid_exists(stdio_pid))
        stdio_tools = await stdio_client.list_tools()
        # 工具说明
        print("stdio_tools 类型: ",type(stdio_tools[0]))
        #stdio_client.call_tool("add",{"a":2,"b":5})
        stdio_add_result = await stdio_client.call_tool("add", {"a": 2, "b": 3})
        stdio_divide_result = await stdio_client.call_tool(
            "divide", {"numerator": 1, "denominator": 3, "precision": 4}
        )
        print("加法:", type(stdio_add_result).__name__, stdio_add_result.data)
        print("除法:", stdio_divide_result.data)

    stdio_schemas = {item.name: item.input_schema for item in stdio_tools}
    stdio_closed = not stdio_client.is_connected() and not psutil.pid_exists(stdio_pid)
    print("退出后 Client 已连接:", stdio_client.is_connected())
    print("退出后服务进程存在:", psutil.pid_exists(stdio_pid))
    print("工具 Schema 与进程内基准一致:", stdio_schemas == baseline_schemas)
    print("实际 stderr 日志:")
    for line in calculator_log_lines(stdio_log_path.read_text(encoding="utf-8")):
        print(line)

进入前 Client 已连接: False
进入后 Client 已连接: True
服务 PID: 85820 进程存在: True
stdio_tools 类型:  <class 'mcp_types._types.Tool'>
加法: CallToolResult 5
除法: 0.3333
退出后 Client 已连接: False
退出后服务进程存在: False
工具 Schema 与进程内基准一致: True
实际 stderr 日志:
calculator: server_pid=85820 transport=stdio
calculator: add: a=2 b=3
calculator: divide: numerator=1.0 denominator=3.0 precision=4


返回的 `CallToolResult.data` 分别是 5 和 0.3333。日志记录了同一服务 PID 的启动与函数执行，而客户端仍能正确解析协议结果，说明日志走的是独立的 stderr 通道。

### 4.4.2 修改一个条件：keep_alive=True会怎样

把 keep_alive 改成 True，两次 `async with` 复用同一个 Client 和 Transport。预测：第一次退出后，Client 是否仍连接？子进程是否仍存在？第二次进入是否创建新 PID？

本实验最后用 `await client.close()` 显式清理。保留进程的配置与“当前 Client 已连接”是两个不同状态。

In [11]:
with TemporaryDirectory(prefix="mcp-stdio-reuse-") as log_directory:
    persistent_log_path = Path(log_directory) / "calculator-stderr.log"
    persistent_transport = PythonStdioTransport(
        script_path=SCRIPT_FILE,
        args=["--transport", "stdio"],
        python_cmd=sys.executable,
        cwd=str(PROJECT_ROOT),
        keep_alive=True,
        log_file=persistent_log_path,
    )
    persistent_client = Client(persistent_transport, timeout=15, init_timeout=15)
    try:
        async with persistent_client:
            await persistent_client.call_tool("add", {"a": 4, "b": 5})
            first_pid = read_server_pid(persistent_log_path)

        print("第一次退出后 Client 已连接:", persistent_client.is_connected())
        persistent_survived = psutil.pid_exists(first_pid)
        print("第一次退出后服务进程存在:", persistent_survived)

        async with persistent_client:
            reused_result = await persistent_client.call_tool("add", {"a": 10, "b": -3})
            second_pid = read_server_pid(persistent_log_path)
            print("第二次调用结果:", reused_result.data)
            persistent_reused = first_pid == second_pid
            print("两次连接复用相同 PID:", persistent_reused)
    finally:
        close_return = await persistent_client.close()

    persistent_closed = not psutil.pid_exists(first_pid)
    print("显式 close 的返回值:", close_return)
    print("显式 close 后服务进程存在:", psutil.pid_exists(first_pid))

第一次退出后 Client 已连接: False
第一次退出后服务进程存在: True
第二次调用结果: 7
两次连接复用相同 PID: True
显式 close 的返回值: None
显式 close 后服务进程存在: False


## 4.5 Streamable HTTP：先启动服务，再连接URL

![Streamable HTTP 的 host、port、path 和 URL](assets/15-streamable-http-url.svg)

本节的自动启动命令等价于：

```python
mcp.run(transport="http", host="127.0.0.1", port=选定端口, path="/mcp")
```

`path` 是服务端的 MCP 端点，客户端必须连接相同路径。URL 中的端口也必须对应当前服务，而不是直接照搬图中的 8000。用普通浏览器 GET 打开 URL，不能代替 MCP 协议的初始化、发现与调用。

### 4.5.1 StreamableHttpTransport的参数和返回值

```python
StreamableHttpTransport(
    url, headers=None, auth=None,
    httpx_client_factory=None, verify=None,
)
```

| 参数 | 含义 |
|---|---|
| `url` | str 或 URL 对象；完整 MCP URL，例如 `http://127.0.0.1:8000/mcp` |
| `headers` | 请求头字符串字典或 None；本地计算服务不需要，使用 None |
| `auth` | 认证配置或 None；本服务没有认证，使用 None |
| `httpx_client_factory` | 自定义底层 HTTP 客户端的工厂函数或 None；本节使用库的默认实现 |
| `verify` | HTTPS 证书校验配置；本节使用本机 HTTP，无需改动 |
| 构造返回值 | StreamableHttpTransport 对象；不启动服务、不返回工具结果 |

把对象交给 `Client(transport)`，进入上下文后才连接服务。`Client(url)` 也能自动识别 HTTP 连接；本节先显式构造 Transport，方便看清各对象的职责。

### 4.5.2 启动HTTP子进程需要哪些普通Python接口

HTTP Transport 不负责启动服务器。为让整本笔记自动运行，本节编写一个启动器管理服务子进程；它模拟在独立终端启动和停止服务的操作。

| 普通 Python 接口 | 参数与返回值 |
|---|---|
| `await asyncio.create_subprocess_exec(program, *args, cwd=..., stdout=..., stderr=...)` | program 为 sys.executable，args 为脚本及命令行选项；cwd 为工作目录。等待后返回 asyncio 的 Process 对象，子进程已被创建 |
| `process.pid` / `process.returncode` | pid 是进程编号；returncode 运行时为 None，结束后为整数；在 POSIX 上负值可表示收到信号而结束 |
| `process.terminate()` | 向服务进程请求结束，同步返回 None；POSIX 上发送 SIGTERM |
| `await process.wait()` | 等待结束，返回退出码；它本身不会请求进程停止 |
| `process.kill()` | 强制结束进程，同步返回 None；本节仅在请求关闭后等待超时才使用 |
| `await asyncio.wait_for(awaitable, timeout)` | 设置等待秒数；正常时返回被等待操作的结果，超时抛出 TimeoutError |

`stdout=log_file, stderr=asyncio.subprocess.STDOUT` 把 HTTP 服务的两条输出流合并到日志文件。本例的 HTTP 协议经网络收发，日志重定向不影响 MCP 通信。

下面三个辅助函数只负责选择端口、等待监听和停止进程。监听探测成功仅表示端口已接受连接，之后仍通过 MCP 工具发现和调用验证协议。

In [ ]:
def choose_available_port(host: str) -> int:
    """让操作系统分配一个本机可用端口，取得编号后关闭探测 socket。"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.bind((host, 0))
        return probe.getsockname()[1]


async def wait_for_listener(process, host: str, port: int, log_path: Path,
                            timeout: float = 15) -> None:
    """等待服务接受连接；服务提前退出或超过指定秒数时报告日志。"""
    #time.monotonic() 用来返回一个单调时钟（monotonic clock）的值
    deadline = time.monotonic() + timeout
    # 当时间
    while time.monotonic() < deadline:
        if process.returncode is not None:
            raise RuntimeError(log_path.read_text(encoding="utf-8"))
        try:
            _, writer = await asyncio.wait_for(
                asyncio.open_connection(host, port), timeout=0.3
            )
        except (OSError, asyncio.TimeoutError):
            await asyncio.sleep(0.05)
        else:
            writer.close()
            await writer.wait_closed()
            return
    raise TimeoutError(log_path.read_text(encoding="utf-8"))


async def stop_server_process(process) -> int:
    """请求停止服务并等待退出，超过 10 秒则强制停止。"""
    if process.returncode is None:
        try:
            process.terminate()
        except ProcessLookupError:
            pass
    try:
        return await asyncio.wait_for(process.wait(), timeout=10)
    except asyncio.TimeoutError:
        process.kill()
        return await process.wait()

### 4.5.3 给HTTP服务进程一个明确的管理范围

![HTTP 服务进程与 Client 会话的两个管理范围](assets/16-http-client-and-server-lifecycle.svg)

`running_http_server(host, path)` 是**我们编写的异步上下文管理器**，不是 FastMCP 内置 API。进入时启动计算脚本并等待监听，返回的状态字典包含 url、process 和日志路径；退出时请求服务关闭、等待退出并清理临时文件。

`@asynccontextmanager` 是 Python 标准库装饰器：yield 前是进入时的准备，yield 提供上下文中的对象，finally 中是退出时的清理。把服务进程放在外层、Client 放在内层，能分别观察两个生命周期。即使工具调用抛错，finally 也会清理服务。

In [13]:
@asynccontextmanager
async def running_http_server(host: str = HTTP_HOST, path: str = MCP_PATH):
    # 返回接口
    port = choose_available_port(host)
    # 将Server挂载到端口启动
    command = [
        sys.executable, "-u", str(SCRIPT_FILE),
        "--transport", "http", "--host", host,
        "--port", str(port), "--path", path,
    ]
    with TemporaryDirectory(prefix="mcp-http-") as log_directory:
        log_path = Path(log_directory) / "calculator-http.log"
        with log_path.open("w", encoding="utf-8") as log_file:
            # 创建异步子进程开启Server
            process = await asyncio.create_subprocess_exec(
                *command,
                cwd=str(PROJECT_ROOT),
                stdout=log_file,
                stderr=asyncio.subprocess.STDOUT,
            )
            state = {
                "url": f"http://{host}:{port}{path}",
                "process": process,
                "log_path": log_path,
            }
            try:
                await wait_for_listener(process, host, port, log_path)
                yield state
            finally:
                state["returncode"] = await stop_server_process(process)
                state["log_text"] = log_path.read_text(encoding="utf-8")

## 4.6 小实验：Client关闭后，HTTP服务还在吗

### 4.6.1 先准备LangChain工具执行的观察函数

沿用第三节的手工 ToolCall：调用 MCPAdapter 返回的工具，检查真实 ToolMessage。`target` 可以是 stdio Transport，也可以是 HTTP URL；异步辅助函数返回一个记录消息类型、结果和状态的 dict。

这验证 Agent 的工具执行部分，ToolCall 由我们填写，没有请求模型。适配后取得的工具列表可以传给 `create_agent(model=..., tools=tools)`。

In [14]:
async def inspect_adapted_tool(target, label: str) -> dict:
    async with MCPAdapter(target) as adapter:
        # 这里是已经包装好的 Tools
        tools = await adapter.list_tools()
        adapted_add = next(item for item in tools if item.name == "add")
        message = await adapted_add.ainvoke({
            "name": "add",
            "args": {"a": 2, "b": 3},
            "id": f"{label}_connection_demo",
            "type": "tool_call",
        })
    return {
        "tool_type": type(adapted_add).__name__,
        "is_tool_message": isinstance(message, ToolMessage),
        "status": message.status,
        "text": message.text,
        "tool_call_id": message.tool_call_id,
    }

### 4.6.2 Client A关闭，Client B再连接同一个服务

外层启动器运行期间，先用显式的 StreamableHttpTransport 连接 A；关闭 A 后检查 Process.returncode，再用 `Client(url)` 连接 B。观察两个客户端是否仍连接同一个服务 PID、是否还可以调用除法。

最后通过 MCPAdapter 验证 HTTP 与 stdio 工具适配，退出外层才停止 HTTP 服务。所有协议调用都在各自的 Client 或 Adapter 上下文内执行。

In [21]:
# 启动服务端
# 上下文管理器会实现  __aenter__ 和 
async with running_http_server() as http_server:
    http_url = http_server["url"]
    http_process = http_server["process"]
    print("已启动 HTTP 服务，PID:", http_process.pid)
    print("实际 MCP URL:", http_url)
    # 返回连接配置
    http_transport = StreamableHttpTransport(url=http_url)
    http_client_a = Client(http_transport, timeout=15, init_timeout=15)
    print("连接前 A 已连接:", http_client_a.is_connected())
    async with http_client_a:
        print("连接后 A 已连接:", http_client_a.is_connected())
        http_tools = await http_client_a.list_tools()
        http_add_result = await http_client_a.call_tool("add", {"a": 2, "b": 3})
        http_divide_result = await http_client_a.call_tool(
            "divide", {"numerator": 1, "denominator": 3, "precision": 4}
        )
        print("HTTP 加法与除法结果:", http_add_result.data, http_divide_result.data)

    http_alive_after_client_close = http_process.returncode is None
    print("A 退出后已连接:", http_client_a.is_connected())
    print("A 退出后 HTTP 服务仍在:", http_alive_after_client_close)

    async with Client(http_url, timeout=15, init_timeout=15) as http_client_b:
        second_http_result = await http_client_b.call_tool(
            "divide", {"numerator": 1, "denominator": 2}
        )
        print("B 再连接同一 PID:", http_process.pid, "计算结果:", second_http_result.data)
    # 将 http_url 连接的客户端调用包装好返回结果
    http_adapted = await inspect_adapted_tool(http_url, "http")
    # 创建使用输入输出通信的配置
    adapter_stdio_transport = PythonStdioTransport(
        script_path=SCRIPT_FILE,
        args=["--transport", "stdio"],
        python_cmd=sys.executable,
        cwd=str(PROJECT_ROOT),
        keep_alive=False,
        log_file=http_server["log_path"].with_name("adapter-stdio.log"),
    )
    # 使用 输入输出 连接的 包装好的工具调用结果
    stdio_adapted = await inspect_adapted_tool(adapter_stdio_transport, "stdio")
    adapter_stdio_pid = read_server_pid(http_server["log_path"].with_name("adapter-stdio.log"))
    print("HTTP 适配结果:", http_adapted)
    print("stdio 适配结果:", stdio_adapted)
    print("stdio Adapter 退出后子进程已关闭:", not psutil.pid_exists(adapter_stdio_pid))

http_schemas = {item.name: item.input_schema for item in http_tools}
http_closed = http_process.returncode is not None and not psutil.pid_exists(http_process.pid)
print("退出外层后 HTTP 服务进程存在:", psutil.pid_exists(http_process.pid))
print("HTTP Process.wait 的退出码:", http_server["returncode"])
print("HTTP 服务的计算日志:")
for line in calculator_log_lines(http_server["log_text"]):
    print(line)

已启动 HTTP 服务，PID: 4575
实际 MCP URL: http://127.0.0.1:64129/mcp
连接前 A 已连接: False
连接后 A 已连接: True
HTTP 加法与除法结果: 5 0.3333
A 退出后已连接: False
A 退出后 HTTP 服务仍在: True
B 再连接同一 PID: 4575 计算结果: 0.5
HTTP 适配结果: {'tool_type': 'StructuredTool', 'is_tool_message': True, 'status': 'success', 'text': '5', 'tool_call_id': 'http_connection_demo'}
stdio 适配结果: {'tool_type': 'StructuredTool', 'is_tool_message': True, 'status': 'success', 'text': '5', 'tool_call_id': 'stdio_connection_demo'}
stdio Adapter 退出后子进程已关闭: True
退出外层后 HTTP 服务进程存在: False
HTTP Process.wait 的退出码: -15
HTTP 服务的计算日志:
calculator: server_pid=4575 transport=http
calculator: add: a=2 b=3
calculator: divide: numerator=1.0 denominator=3.0 precision=4
calculator: divide: numerator=1.0 denominator=2.0 precision=2
calculator: add: a=2 b=3


关闭 A 后，HTTP 服务继续运行，B 和 MCPAdapter 都可以重新连接同一 URL。退出外层启动器后，服务才结束。stdio Adapter 则按 keep_alive=False 关闭自己启动的子进程。

在本机的 POSIX 环境中，退出码 -15 表示进程因 SIGTERM 结束，正是启动器发出的关闭请求。实验通过 returncode 不再为 None、进程不再存在确认关闭完成。

### 4.6.3 再修改一个条件：换HTTP端点路径

把路径改成 `/calculator`，端口由启动器重新选择，工具定义保持不变。使用启动器返回的新 URL 连接；不再使用上一轮的旧 URL。

In [23]:
async with running_http_server(path="/calculator") as changed_server:
    print("新的 MCP URL:", changed_server["url"])
    async with Client(changed_server["url"], timeout=15, init_timeout=15) as changed_client:
        changed_tools = await changed_client.list_tools()
        changed_result = await changed_client.call_tool("add", {"a": 10, "b": -3})
        print("新的参数计算结果:", changed_result.data)
        print("路径改变后 Schema 仍一致:",
              {item.name: item.input_schema for item in changed_tools} == baseline_schemas)
print("第二个 HTTP 服务已退出:", changed_server["process"].returncode is not None)

新的 MCP URL: http://127.0.0.1:65287/calculator
新的参数计算结果: 7
路径改变后 Schema 仍一致: True
第二个 HTTP 服务已退出: True


## 4.7 核对两种连接的共同点与差异

换的是传输方式，计算函数及参数格式保持不变。核对实际结果与清理状态：

In [24]:
checks = {
    "stdio Schema 与进程内一致": stdio_schemas == baseline_schemas,
    "HTTP Schema 与进程内一致": http_schemas == baseline_schemas,
    "两种连接的加法结果都是 5": stdio_add_result.data == http_add_result.data == 5,
    "两种连接的除法结果都是 0.3333":
        stdio_divide_result.data == http_divide_result.data == 0.3333,
    "stdio keep_alive=False 已结束子进程": stdio_closed,
    "stdio keep_alive=True 保留并复用进程": persistent_survived and persistent_reused,
    "stdio keep_alive=True 显式 close 后已结束": persistent_closed,
    "HTTP Client 关闭后服务曾继续运行": http_alive_after_client_close,
    "HTTP 外层启动器退出后服务已结束": http_closed,
    "两种适配都返回成功 ToolMessage": all(
        item["is_tool_message"] and item["status"] == "success" and item["text"] == "5"
        for item in [stdio_adapted, http_adapted]
    ),
    "路径修改实验已结束服务": changed_server["process"].returncode is not None,
    "stdio Adapter 的子进程已结束": not psutil.pid_exists(adapter_stdio_pid),
}
for label, passed in checks.items():
    print(f"{label}: {passed}")
assert all(checks.values()), checks

stdio Schema 与进程内一致: True
HTTP Schema 与进程内一致: True
两种连接的加法结果都是 5: True
两种连接的除法结果都是 0.3333: True
stdio keep_alive=False 已结束子进程: True
stdio keep_alive=True 保留并复用进程: True
stdio keep_alive=True 显式 close 后已结束: True
HTTP Client 关闭后服务曾继续运行: True
HTTP 外层启动器退出后服务已结束: True
两种适配都返回成功 ToolMessage: True
路径修改实验已结束服务: True
stdio Adapter 的子进程已结束: True


| 动作 | stdio，keep_alive=False | stdio，keep_alive=True | Streamable HTTP |
|---|---|---|---|
| 构造 Transport | 保存脚本启动配置 | 保存脚本启动配置 | 保存 URL 连接配置 |
| 进入 Client | 启动子进程并初始化 MCP 连接 | 首次启动，后续可复用子进程 | 连接已经运行的服务 |
| 退出 Client 上下文 | 关闭连接与子进程 | Client 退出，子进程保留供复用 | Client 退出，独立服务继续运行 |
| 显式停止进程 | 通常已由 Transport 完成 | `await client.close()` 完成底层清理 | 由服务运行者停止，本节外层启动器请求停止并等待退出 |
| 工具调用返回 | CallToolResult | CallToolResult | CallToolResult |
| 交给 Agent | MCPAdapter 适配后的工具 | 同左 | 同左 |

## 4.8 小练习

1. 在 4.4.1 的 stdio 调用里把精度改成 2，再改成 6。查看业务结果与 stderr 日志；不用修改连接配置。
2. 在 4.4.2 增加第三次 `async with persistent_client`，观察 PID 是否继续复用，再确认显式 close 后进程结束。
3. 把 4.6.3 的路径改成 `/calc`，使用启动器返回的 URL。解释为什么修改服务端 path 后也要修改客户端 URL。
4. 在独立终端按 4.3 的 HTTP 命令启动服务，用 notebook 新单元格的 Client 连接它。退出 Client 后观察终端服务仍在，最后在该终端按 Ctrl+C 关闭。

下一节继续复用 FastMCP，区分 Tools、Resources 和 Prompts 的发现、获取与使用。

参考：[FastMCP 服务启动](https://gofastmcp.com/deployment/running-server)、[FastMCP Client Transports](https://gofastmcp.com/clients/transports)、[MCP 传输规范](https://modelcontextprotocol.io/specification/2025-11-25/basic/transports)、[Python 异步子进程](https://docs.python.org/3.11/library/asyncio-subprocess.html)。参数与输出以本节打印的环境版本为准。